# Hour 5 — Webhooks: `dalux.webhook_server`

This section turns the notebook into a local test harness for the embedded scheduled monitor. It covers
`start()`, `register_change_job()`, `register_freshness_job()`, manually triggering a job with
`test_job()` / `test_webhook()`, the periodic job-snapshot log, and callback delivery logging.

> Dalux Build itself does not push webhooks to you — there is no server-side subscription API. `webhook_server`
> closes that gap client-side: it runs wherever you run this code, polls on your behalf on a schedule you define,
> and delivers a webhook-style HTTP callback to an endpoint you control.

> **Heads up:** `test_job()` / `test_webhook()` and the built-in job-snapshot / delivery-success logging are
> landing in an upcoming `dalux_build` release. If you're pinned to an older release you won't see them yet —
> bump the `dalux-build[webhook]` pin in `pyproject.toml` once it's out.

**By the end of this hour you will be able to:**

- Install the `webhook` extra and configure `MONITOR_API_TOKEN` / `MONITOR_MASTER_KEY`
- Start the embedded monitor with `dalux.webhook_server.start()`
- Register a **change job** that calls you back when files in a file area change
- Register a **freshness job** that calls you back if a set of models hasn't been updated in N days (e.g. once a
  week)
- Trigger a manual test run with `test_job()` before waiting on a real schedule
- Watch job snapshots and delivery status through the Python logger
- Unregister jobs and stop the monitor cleanly

## Local test flow

- Install the optional `webhook` extra if it is not already present.
- Reconnect to your project (same pattern as Hours 1–4).
- Start the embedded monitor with local secrets and turn on logging.
- Register a change job and a freshness job.
- Trigger a manual test webhook before you wait for a scheduled run.
- Watch the notebook output and the Python logger output for job snapshots and delivery status.

## 0. Reconnect

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Works whether Jupyter was launched from the repo root or from tutorials/
for candidate in (Path(".env"), Path("../.env")):
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()  # fall back to variables already exported in the shell

import os
assert os.getenv("DALUX_API_KEY"), "DALUX_API_KEY not found — copy .env.example to .env and fill it in"
assert os.getenv("DALUX_BASE_URL"), "DALUX_BASE_URL not found — copy .env.example to .env and fill it in"
print("DALUX_BASE_URL:", os.getenv("DALUX_BASE_URL"))

In [ ]:
from dalux_build import create_client

dalux = create_client()
dalux

In [ ]:
# Pick the first project in your account to work with for the rest of this notebook.
# Swap this for dalux.projects.get_project_by_name("Your Project Name") if you want a specific one.
projects_response = dalux.projects.list_projects()

PROJECT_ID = projects_response[0].project_id
print("Using project:", projects_response[0].project_name, f"({PROJECT_ID})")

dalux.set_default_project(PROJECT_ID)

In [ ]:
# Several charts/jobs in this hour need a file_area_id too — reuse the Hour 2 pattern.
file_areas_response = dalux.file_areas.get_file_areas()

FILE_AREA_ID = file_areas_response[0].file_area_id
FILE_AREA_NAME = file_areas_response[0].file_area_name
print("Using file area:", FILE_AREA_NAME, f"({FILE_AREA_ID})")

dalux.set_default_file_area(FILE_AREA_ID)

## 1. Install the `webhook` extra

`webhook_server` needs `fastapi`, `uvicorn`, `httpx`, `cryptography` and `croniter` — kept as an optional extra so
the base `dalux_build` install stays light for people who never touch monitoring.

In [ ]:
!uv add "dalux-build[webhook]" -qU

## 2. Secrets, logging & a local callback receiver

- `MONITOR_API_TOKEN` — a bearer token that protects the monitor's own local REST API (the endpoints it exposes
  for creating/deleting/testing jobs). Any sufficiently random string works.
- `MONITOR_MASTER_KEY` — a [Fernet](https://cryptography.io/en/latest/fernet/) key used to encrypt your Dalux API
  key and any callback secrets before the monitor writes them to its local SQLite state file. This one **must**
  be a valid Fernet key, not just any random string.

The cell below generates both once per kernel session (`os.environ.setdefault`, so re-running it won't rotate
secrets out from under a monitor that's already running) — no manual `.env` copy-paste or kernel restart needed.
It also turns on INFO-level logging for `dalux_build.webhook_server` and its `scheduler` / `delivery` / `monitor`
submodules so you can watch job snapshots and delivery attempts scroll by, and spins up a minimal local HTTP
server so you can see callbacks arrive in this notebook.

In [ ]:
import json
import logging
import os
import secrets
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer

from cryptography.fernet import Fernet

logging.basicConfig(level=logging.INFO)
for logger_name in (
    "dalux_build.webhook_server",
    "dalux_build.webhook_server.scheduler",
    "dalux_build.webhook_server.delivery",
    "dalux_build.webhook_server.monitor",
):
    logging.getLogger(logger_name).setLevel(logging.INFO)

os.environ.setdefault("MONITOR_API_TOKEN", secrets.token_urlsafe(32))
os.environ.setdefault("MONITOR_MASTER_KEY", Fernet.generate_key().decode())

received = []


class CallbackHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"{}")
        received.append(payload)
        print("Callback received:", payload)
        self.send_response(200)
        self.end_headers()

    def log_message(self, *args):
        pass  # silence default request logging


callback_server = HTTPServer(("127.0.0.1", 8003), CallbackHandler)
threading.Thread(target=callback_server.serve_forever, daemon=True).start()

CALLBACK_URL = "http://127.0.0.1:8003/"
print("Listening for callbacks on", CALLBACK_URL)
print("MONITOR_API_TOKEN:", os.environ["MONITOR_API_TOKEN"])
print("MONITOR_MASTER_KEY:", os.environ["MONITOR_MASTER_KEY"][:16] + "...")

## 3. Starting the monitor — `dalux.webhook_server.start()`

`start()` will happily read `MONITOR_API_TOKEN`, `MONITOR_MASTER_KEY` and `DALUX_BASE_URL` straight from the
environment, or you can pass them explicitly as below (handy since we just generated them in code, above). It
binds a local FastAPI/uvicorn server and owns a small SQLite state file plus a background scheduler thread — jobs
survive a restart of the monitor as long as the state file is kept.

In [ ]:
dalux.webhook_server.start(
    management_token=os.environ["MONITOR_API_TOKEN"],
    master_key=os.environ["MONITOR_MASTER_KEY"],
    host="127.0.0.1",
    state_db_path="./hour_5_monitor_state.sqlite3",
)

print("Monitor running:", dalux.webhook_server.is_running)
print("Monitor URL:", dalux.webhook_server.url)

## 4. Change job — call me when files change

`register_change_job` polls a file area on a cron schedule and calls your `callback_url` when it detects added,
modified or removed files.

- `scope="all"` watches every file in the file area; `scope="fileIds"` + `file_ids=[...]` watches specific files
- `initial_run="baseline"` records the current state on the first poll without firing a callback for it;
  `"emitCurrent"` fires once immediately for every file that already exists

In [ ]:
change_job_id = dalux.webhook_server.register_change_job(
    project_id=PROJECT_ID,
    file_area_id=FILE_AREA_ID,
    cron="*/15 * * * *",       # check every 15 minutes
    callback_url=CALLBACK_URL,
    scope="all",                # or scope="fileIds", file_ids=[...] to watch specific files
    initial_run="baseline",     # only fire for changes from now on, not for files that already exist
    name="hour-5-demo-change-job",
)
print("Registered change job:", change_job_id)

## 5. Freshness job — call me if a set of models hasn't been updated in a week

This is the "once a week" job from the Hour 1 roadmap: instead of watching for *any* change, `register_freshness_job`
periodically checks whether files matching a `FileNameFilter` (e.g. your Revit/IFC models) have been modified more
recently than `max_age` (an ISO-8601 duration like `"P7D"` for 7 days) — and calls you back with the stale ones.
The `cron` here controls how often you *check*, not how often the models need to change.

In [ ]:
from dalux_build.models import FileNameFilter

freshness_job_id = dalux.webhook_server.register_freshness_job(
    project_id=PROJECT_ID,
    file_area_id=FILE_AREA_ID,
    cron="0 8 * * 1",                                              # check every Monday at 08:00
    callback_url=CALLBACK_URL,
    file_name_filter=FileNameFilter(extensions=["rvt", "ifc"]),    # the "set of models" you care about
    max_age="P7D",                                                  # flag files not modified in the last 7 days
    name="hour-5-demo-freshness-job",
)
print("Registered freshness job:", freshness_job_id)

## 6. Authenticating callbacks

By default callbacks are unauthenticated (`callback_auth_type="none"`). For a real receiver, pass
`callback_auth_type="bearer"` (a static `Authorization: Bearer <secret>` header) or `"hmac-sha256"` (the body is
signed with your secret so the receiver can verify it wasn't spoofed), plus a `callback_secret`.

In [ ]:
signed_job_id = dalux.webhook_server.register_change_job(
    project_id=PROJECT_ID,
    file_area_id=FILE_AREA_ID,
    cron="*/15 * * * *",
    callback_url=CALLBACK_URL,
    scope="all",
    callback_auth_type="bearer",
    callback_secret="a-secret-only-you-and-your-receiver-know",
    name="hour-5-demo-signed-job",
)
dalux.webhook_server.unregister_job(signed_job_id)  # this cell is illustrative only — clean up immediately

## 7. Triggering a manual test — `test_job()` / `test_webhook()`

Rather than waiting for the cron schedule, `test_job()` (aliased as `test_webhook()`) fires a job's callback
immediately with its current state — handy for confirming your receiver and auth are wired up correctly before
you trust a scheduled run. Watch the logger output from the setup cell in section 2 alongside the callback
printed below.

In [ ]:
test_response = dalux.webhook_server.test_job(change_job_id)
print(test_response)

# test_webhook() is kept as an alias for the same behavior.
print(dalux.webhook_server.test_webhook(change_job_id))
print("Callbacks received:", len(received))

## 8. Cleaning up — unregister jobs and stop the monitor

Unregister the demo jobs and stop the monitor (and the local callback receiver from step 3) when you're done —
in a real deployment you'd instead leave `dalux.webhook_server` running as a long-lived process.

In [ ]:
dalux.webhook_server.unregister_job(change_job_id)
dalux.webhook_server.unregister_job(freshness_job_id)

dalux.webhook_server.stop()
callback_server.shutdown()

print("Monitor running:", dalux.webhook_server.is_running)

## 9. Error handling

`webhook_server` raises its own small exception hierarchy, importable from `dalux_build.webhook_server`:

- `WebhookServerError` — base class for everything below
- `WebhookServerAlreadyRunning` — raised by `start()` if the monitor is already running
- `WebhookServerNotRunning` — raised by `register_*_job()` / `unregister_job()` before `start()` has been called
- `MissingWebhookDependencies` — raised (as an `ImportError` subclass) if the `webhook` extra isn't installed


## Recap — the full tutorial

- **Hour 1:** credentials, `create_client()`, the `DaluxClient` namespaces, projects/companies/users, error handling
- **Hour 2:** file areas → folders → files, tree building, downloads (single/bulk/filtered), chunked uploads
- **Hour 3:** tasks, task change history, forms, work packages, the other read-only resources, and the
 `find_by_field` utilities that generalize to any endpoint
- **Hour 4:** charting `to_dataframe=True` results with `matplotlib`, and combining charts into a dashboard
- **Hour 5 (this one):** `dalux.webhook_server` — change jobs, freshness jobs, manual test triggers, and
 job-snapshot/delivery logging, so you don't have to poll the API yourself or fly blind while testing

For the full method-by-method reference, see the `dalux_build` package's own README (or ask Claude Code — this
repo includes a `dalux-build` skill under `.claude/skills/` that documents every API class and method for quick
lookups while you build).